# CornerNet（PASCAL VOC 2007を用いた物体検出）

---
## 目的
これまでのSSDなどの物体検出手法は，画像上に大量の「デフォルトボックス（アンカー）」を敷き詰め，各アンカーに対してクラスと位置のずれを回帰する，**アンカーベース**の手法でした．本ノートブックで扱うCornerNetは，これとは全く異なるアプローチを取る**アンカーフリー**の物体検出手法です．

CornerNetは，物体検出を「矩形の左上（top-left）と右下（bottom-right）という2つのキーポイント（コーナー）をペアで検出する問題」として定式化します．ネットワークは，クラスごとの左上コーナー・右下コーナーの位置を表す**ヒートマップ**を出力し，さらに同じ物体に属する2つのコーナーを対応付けるための**embedding（埋め込みベクトル）**を出力します．アンカーの設計（アスペクト比・スケールの選定）が不要になる一方で，ヒートマップからのピーク検出やコーナーのペアリングといった，SSDとは異なる推論の仕組みが必要になります．本ノートブックでは，このヒートマップベースの検出パラダイムと，CornerNet特有の「Corner Pooling」という演算について理解することを目的とします．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
get_ipython().system('pip install -q torchmetrics')

import os
import math
import zipfile
from time import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．データ読み込み部分は`ssd.ipynb`と共通のため，同じ実装を流用します（入力画像サイズ`IMG_SIZE`のみ変更しています）．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる（ssd.ipynbと同じ規約）
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数（データセット側のラベル規約に合わせるために保持）
IMG_SIZE = 256  # CornerNetは出力ストライドが小さい（解像度の高い）ヒートマップを使うため，SSDより小さめの入力サイズにする

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    '''VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する'''
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    '''VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset'''
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む，データセット側の規約）:', n_class)

## CornerNet: バックボーンとアップサンプリング
原論文のCornerNetは，姿勢推定などでも使われる**Stacked Hourglass Network**をバックボーンとして使用します．Hourglassネットワークはダウンサンプリングとアップサンプリングを繰り返す砂時計型の構造を持ち，スクラッチから学習されます．しかし本シリーズの方針に従い，本ノートブックではImageNet事前学習済みのResNet50をバックボーンとして使用します（この置き換えは本シリーズの検出ノートブックの中でも特に大きな変更であるため，詳細は末尾の「オリジナル論文との違い」で改めて説明します）．

ヒートマップベースの検出では，ヒートマップ上のピーク位置そのものが検出結果の座標になるため，SSDのようなアンカーベースの手法よりもずっと高い空間解像度の特徴マップが必要です．そこで，ResNet50の`layer4`（入力に対してストライド32）の出力に対して，ストライド2の転置畳み込み（`nn.ConvTranspose2d`）を3層重ね，ストライド4（入力画像の1/4の解像度）まで解像度を戻します．この「分類用バックボーン + Deconv層によるアップサンプリング」という構成は，Hourglass以外のバックボーンでCornerNetやCenterNetを実装する際の実装例（例えばCenterNetの公式実装のResNetベース版）でも実際に使われている一般的なパターンです．

`IMG_SIZE = 256`のとき，最終的な特徴マップの解像度は`256 / 4 = 64`（64×64）になります．

In [ ]:
STRIDE = 4  # バックボーン出力の入力画像に対するストライド（解像度は 1/4 ）
HEATMAP_SIZE = IMG_SIZE // STRIDE
NUM_HEATMAP_CLASSES = len(VOC_CLASSES)  # 背景の明示的なchannelは持たない（詳細は後述のヘッドの節を参照）


class CornerNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4)  # stride 32, channels 2048

        # stride 32 -> 16 -> 8 -> 4 と，転置畳み込みで段階的に解像度を戻す
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(2048, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.out_channels = 256

    def forward(self, x):
        x = self.stem(x)
        x = self.deconv(x)
        return x


# 動作確認：入力画像サイズに対してストライド4の特徴マップが得られるか確認する
_backbone_check = CornerNetBackbone().to(device)
_dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    _out = _backbone_check(_dummy)
print('backbone output shape:', _out.shape, '(期待値: 1,', _backbone_check.out_channels, ',', HEATMAP_SIZE, ',', HEATMAP_SIZE, ')')
del _backbone_check, _dummy, _out

## Corner Pooling
物体の左上コーナーの位置には，多くの場合，その物体を示す明確な特徴（エッジなど）が存在するとは限りません（コーナーの内側は物体，外側は背景であるだけで，コーナー自体には模様がないことが多い）．そこでCornerNetは，コーナー位置の特徴を「その物体の上端」と「その物体の左端」の情報から作り出す，**Corner Pooling**という演算を提案しています．

- **top-left corner pooling**：特徴マップ上の各位置$(x, y)$について，「同じ行で右側にある全ての位置の最大値」と「同じ列で下側にある全ての位置の最大値」を合計する．物体の左上コーナーは，右方向に物体の上端，下方向に物体の左端が続いているため，これらの最大値を集約することでコーナーらしさを強調できる．
- **bottom-right corner pooling**：左上とは逆に，「同じ行で左側にある全ての位置の最大値」と「同じ列で上側にある全ての位置の最大値」を合計する．

「ある位置から右側（または下側）の最大値」は，`torch.cummax`を使うと効率的に計算できます．`torch.cummax`は「先頭から現在位置までの累積最大値」を計算するため，右側や下側からの累積最大値が欲しい場合は，テンソルを反転（`flip`）してから`cummax`を適用し，最後にもう一度反転して元の向きに戻します．`torchvision.ops`にはこれに相当する演算が用意されていないため，ここは自前で実装します．

In [ ]:
def right_pool(x):
    '''各位置について，同じ行内で右側（自分自身を含む）の最大値を返す'''
    return torch.flip(torch.cummax(torch.flip(x, dims=[3]), dim=3)[0], dims=[3])


def down_pool(x):
    '''各位置について，同じ列内で下側（自分自身を含む）の最大値を返す'''
    return torch.flip(torch.cummax(torch.flip(x, dims=[2]), dim=2)[0], dims=[2])


def left_pool(x):
    '''各位置について，同じ行内で左側（自分自身を含む）の最大値を返す'''
    return torch.cummax(x, dim=3)[0]


def up_pool(x):
    '''各位置について，同じ列内で上側（自分自身を含む）の最大値を返す'''
    return torch.cummax(x, dim=2)[0]


def corner_pool_tl(x):
    '''top-left corner pooling: 右方向の最大値 + 下方向の最大値'''
    return right_pool(x) + down_pool(x)


def corner_pool_br(x):
    '''bottom-right corner pooling: 左方向の最大値 + 上方向の最大値'''
    return left_pool(x) + up_pool(x)


# 動作確認：対角線上に値を置いたトイテンソルでCorner Poolingの効果を確認する
toy = torch.tensor([[[[1., 0., 0., 0.],
                       [0., 2., 0., 0.],
                       [0., 0., 3., 0.],
                       [0., 0., 0., 4.]]]])
print('元のテンソル:\n', toy[0, 0])
print('top-left corner pooling:\n', corner_pool_tl(toy)[0, 0])
print('bottom-right corner pooling:\n', corner_pool_br(toy)[0, 0])

## 検出ヘッド
アップサンプリングされた特徴マップから，top-leftコーナー用・bottom-rightコーナー用それぞれのヘッド（`CornerHead`）で，3種類の出力を予測します．いずれのヘッドも，専用のCorner Pooling（top-leftヘッドは`corner_pool_tl`，bottom-rightヘッドは`corner_pool_br`）を適用してから，3つの分岐に分かれます．

1. **ヒートマップ**：クラスごとのコーナーらしさを表す，`NUM_HEATMAP_CLASSES`（=20）チャンネルのマップ．SSDと異なり，「背景」を表す明示的なチャンネルは用意しません．背景の位置は単に「どのクラスのヒートマップにもピークが立たない」ことで表現されます．出力には`sigmoid`を適用し，各画素・各クラスを独立した2値分類（コーナーか否か）とみなします．
2. **embedding**：1チャンネルのスカラー値．同じ物体の左上・右下コーナーの値が近くなるように学習することで，2つのヒートマップ間でどのコーナー同士が対応するかを判別するために使う．
3. **offset**：2チャンネル（x, y方向）．バックボーンの出力ストライド（4）により失われるサブピクセルの位置精度を補正するためのオフセット．

ヒートマップの最終層は，学習初期には正例（コーナー位置）が画素全体のごく一部しかなく，Focal Lossが不安定になりやすいため，出力（sigmoid適用前）が負に偏るようbiasを初期化します（CornerNetやCenterNetで一般的に使われる初期化テクニックです）．

In [ ]:
class CornerHead(nn.Module):
    def __init__(self, in_channels, pool_fn):
        super().__init__()
        self.pool_fn = pool_fn
        self.pre_conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.post_conv = nn.Sequential(
            nn.Conv2d(128, in_channels, kernel_size=3, padding=1), nn.BatchNorm2d(in_channels))
        self.relu = nn.ReLU(inplace=True)

        self.heatmap_conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, NUM_HEATMAP_CLASSES, kernel_size=1))
        self.embedding_conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, kernel_size=1))
        self.offset_conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 2, kernel_size=1))

        self.heatmap_conv[-1].bias.data.fill_(-2.19)  # 学習初期のFocal Lossを安定させるための負バイアス初期化

    def forward(self, x):
        feat = self.pre_conv(x)
        pooled = self.pool_fn(feat)
        feat = self.relu(self.post_conv(pooled) + x)  # 入力特徴とのスキップ接続

        heatmap = torch.sigmoid(self.heatmap_conv(feat))
        embedding = self.embedding_conv(feat)
        offset = self.offset_conv(feat)
        return heatmap, embedding, offset


class CornerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CornerNetBackbone()
        self.tl_head = CornerHead(self.backbone.out_channels, corner_pool_tl)
        self.br_head = CornerHead(self.backbone.out_channels, corner_pool_br)

    def forward(self, x):
        feat = self.backbone(x)
        tl_heatmap, tl_embedding, tl_offset = self.tl_head(feat)
        br_heatmap, br_embedding, br_offset = self.br_head(feat)
        return tl_heatmap, br_heatmap, tl_embedding, br_embedding, tl_offset, br_offset

## 学習ターゲットの生成
学習時には，1枚の画像に含まれる各GTボックスから，(1) ヒートマップの教師信号，(2) offsetの教師信号，(3) embeddingを取り出す位置（インデックス），を作成する必要があります．

ヒートマップの教師信号は，GTコーナー位置に1を立てた単なる「one-hotマップ」ではなく，コーナー位置を中心とした**ガウス分布**を書き込みます．ネットワークの予測が真の位置から多少ずれていても，そのずれが小さければ小さいペナルティで済むようにするためです．ガウス分布の広がり（半径）は，GTボックスが十分な重なり（IoU）を保てる範囲になるよう，ボックスの大きさから幾何学的に計算します（`gaussian_radius`関数．CornerNet原論文で提案された式をそのまま使用）．

また，1枚の画像あたりの物体数はまちまちなので，バッチ化のために物体数の上限`MAX_OBJS`を決め，それより少ない分はマスク（`reg_mask`）で無効値として扱います．

In [ ]:
MAX_OBJS = 32  # 1画像あたりに扱うGTボックス数の上限（VOCの1画像あたりの物体数を十分カバーする値）


def gaussian_radius(height, width, min_overlap=0.7):
    '''GTボックスの大きさから，最低限のIoUを保てるガウス分布の半径を求める（CornerNet原論文の式）'''
    a1 = 1
    b1 = height + width
    c1 = width * height * (1 - min_overlap) / (1 + min_overlap)
    r1 = (b1 - math.sqrt(b1 ** 2 - 4 * a1 * c1)) / (2 * a1)

    a2 = 4
    b2 = 2 * (height + width)
    c2 = (1 - min_overlap) * width * height
    r2 = (b2 - math.sqrt(b2 ** 2 - 4 * a2 * c2)) / (2 * a2)

    a3 = 4 * min_overlap
    b3 = -2 * min_overlap * (height + width)
    c3 = (min_overlap - 1) * width * height
    r3 = (b3 + math.sqrt(b3 ** 2 - 4 * a3 * c3)) / (2 * a3)

    return max(0, int(min(r1, r2, r3)))


def draw_gaussian(heatmap, center_x, center_y, radius):
    '''heatmap（1クラス分，H×WのNumPy配列）上の(center_x, center_y)を中心に，半径radiusのガウス分布を書き込む（既存値との最大値を採用）'''
    diameter = 2 * radius + 1
    sigma = diameter / 6
    yy, xx = np.ogrid[-radius:radius + 1, -radius:radius + 1]
    gaussian = np.exp(-(xx * xx + yy * yy) / (2 * sigma * sigma + 1e-8))
    gaussian[gaussian < np.finfo(gaussian.dtype).eps * gaussian.max()] = 0

    height, width = heatmap.shape
    left, right = min(center_x, radius), min(width - center_x, radius + 1)
    top, bottom = min(center_y, radius), min(height - center_y, radius + 1)

    masked_heatmap = heatmap[center_y - top:center_y + bottom, center_x - left:center_x + right]
    masked_gaussian = gaussian[radius - top:radius + bottom, radius - left:radius + right]
    if min(masked_heatmap.shape) > 0 and min(masked_gaussian.shape) > 0:
        np.maximum(masked_heatmap, masked_gaussian, out=masked_heatmap)


def build_targets(boxes, labels, heatmap_size=None, stride=STRIDE, max_objs=MAX_OBJS):
    '''1枚の画像のGTボックス（ピクセル座標，xyxy）から学習ターゲットを作成する'''
    heatmap_size = heatmap_size or HEATMAP_SIZE
    tl_heatmap = np.zeros((NUM_HEATMAP_CLASSES, heatmap_size, heatmap_size), dtype=np.float32)
    br_heatmap = np.zeros((NUM_HEATMAP_CLASSES, heatmap_size, heatmap_size), dtype=np.float32)
    tl_offset = np.zeros((max_objs, 2), dtype=np.float32)
    br_offset = np.zeros((max_objs, 2), dtype=np.float32)
    tl_ind = np.zeros((max_objs,), dtype=np.int64)
    br_ind = np.zeros((max_objs,), dtype=np.int64)
    reg_mask = np.zeros((max_objs,), dtype=np.float32)

    num_objs = min(len(boxes), max_objs)
    for k in range(num_objs):
        x1, y1, x2, y2 = boxes[k]
        cls = int(labels[k]) - 1  # 背景を含まないchannel index（0〜19）に変換

        # ヒートマップ解像度（ストライドstride）上の座標に変換
        x1f, y1f, x2f, y2f = x1 / stride, y1 / stride, x2 / stride, y2 / stride
        x1i = min(max(int(x1f), 0), heatmap_size - 1)
        y1i = min(max(int(y1f), 0), heatmap_size - 1)
        x2i = min(max(int(x2f), 0), heatmap_size - 1)
        y2i = min(max(int(y2f), 0), heatmap_size - 1)

        radius = gaussian_radius(y2f - y1f, x2f - x1f)
        draw_gaussian(tl_heatmap[cls], x1i, y1i, radius)
        draw_gaussian(br_heatmap[cls], x2i, y2i, radius)

        tl_offset[k] = [x1f - x1i, y1f - y1i]  # 整数化により失われたサブピクセル成分
        br_offset[k] = [x2f - x2i, y2f - y2i]
        tl_ind[k] = y1i * heatmap_size + x1i  # 予測テンソルからembedding/offsetを取り出すための1次元インデックス
        br_ind[k] = y2i * heatmap_size + x2i
        reg_mask[k] = 1.0

    return tl_heatmap, br_heatmap, tl_offset, br_offset, tl_ind, br_ind, reg_mask


def build_batch_targets(targets):
    '''collate_fnが返すtargetsのリスト（バッチ分）から，学習ターゲットのバッチTensorを作成する'''
    tl_heats, br_heats, tl_offs, br_offs, tl_inds, br_inds, reg_masks = [], [], [], [], [], [], []
    for t in targets:
        result = build_targets(t['boxes'].numpy(), t['labels'].numpy())
        tl_heats.append(result[0]); br_heats.append(result[1])
        tl_offs.append(result[2]); br_offs.append(result[3])
        tl_inds.append(result[4]); br_inds.append(result[5])
        reg_masks.append(result[6])

    def to_tensor(arr_list, dtype):
        return torch.tensor(np.stack(arr_list), dtype=dtype, device=device)

    return (
        to_tensor(tl_heats, torch.float32), to_tensor(br_heats, torch.float32),
        to_tensor(tl_offs, torch.float32), to_tensor(br_offs, torch.float32),
        to_tensor(tl_inds, torch.long), to_tensor(br_inds, torch.long),
        to_tensor(reg_masks, torch.float32),
    )


# 動作確認：1枚分のダミーボックスでターゲットの形状とヒートマップの最大値（=1になるはず）を確認する
_dummy_boxes = np.array([[20.0, 20.0, 120.0, 150.0]])
_dummy_labels = np.array([3])
_res = build_targets(_dummy_boxes, _dummy_labels)
print('tl_heatmap shape:', _res[0].shape, ' max:', _res[0].max())
del _dummy_boxes, _dummy_labels, _res

## 損失関数
CornerNetの損失は，次の4つの要素の重み付き和で構成されます．

- **ヒートマップ損失**：ピクセルごとの2値分類（コーナーか否か）に対する損失ですが，通常のFocal Lossとは異なり，正例の周囲（ガウス分布で値を埋めた範囲）の負例に対するペナルティを弱める，**penalty-reduced focal loss**を使用します．GTヒートマップの値が1に近い（正解コーナーに近い）負例ほど，予測が多少高くても罰則を弱くすることで，密集したコーナー付近での学習を安定させます．
- **Pull損失**：同じ物体の左上・右下コーナーのembeddingの値を近づける（2つの値の平均からの二乗誤差）．
- **Push損失**：異なる物体同士のembeddingの値を，マージン以上離す（Hinge Loss）．PullとPushをあわせて**Associative Embedding**と呼びます．
- **Offset損失**：GTコーナー位置におけるoffset予測値と，整数化により失われたサブピクセル成分との誤差（Smooth L1損失）．

Pull・Push・Offsetの各損失は，`reg_mask`でパディング部分（物体が存在しない位置）を除外して計算します．また，embeddingとoffsetの予測値（`(B, C, H, W)`のテンソル）から，GTコーナー位置に対応する値だけを取り出すために`gather_feature`関数を用います．

In [ ]:
def gather_feature(feat, ind):
    '''feat: (B, C, H, W)から，indで指定した空間位置（1次元インデックス）のC次元ベクトルを取り出す -> (B, max_objs, C)'''
    b, c, h, w = feat.shape
    feat = feat.view(b, c, h * w).permute(0, 2, 1)  # (B, H*W, C)
    ind = ind.unsqueeze(-1).expand(-1, -1, c)  # (B, max_objs, C)
    return feat.gather(1, ind)


def focal_loss(pred, gt):
    '''penalty-reduced focal loss（CornerNet/CenterNetで使われるヒートマップ損失）'''
    pred = pred.clamp(min=1e-6, max=1 - 1e-6)
    pos_mask = gt.eq(1).float()
    neg_mask = gt.lt(1).float()
    neg_weight = (1 - gt).pow(4)  # GTヒートマップの値が1に近い負例ほどペナルティを弱める

    pos_loss = torch.log(pred) * (1 - pred).pow(2) * pos_mask
    neg_loss = torch.log(1 - pred) * pred.pow(2) * neg_weight * neg_mask

    num_pos = pos_mask.sum()
    loss = -(pos_loss.sum() + neg_loss.sum())
    return loss / num_pos.clamp(min=1)


def pull_push_loss(tl_emb, br_emb, tl_ind, br_ind, reg_mask, margin=1.0):
    '''Associative Embeddingの Pull損失・Push損失'''
    tl_e = gather_feature(tl_emb, tl_ind).squeeze(-1)  # (B, max_objs)
    br_e = gather_feature(br_emb, br_ind).squeeze(-1)
    mean_e = (tl_e + br_e) / 2

    # Pull損失：同一物体の2つのコーナーのembeddingを，その平均値に近づける
    pull = (tl_e - mean_e).pow(2) + (br_e - mean_e).pow(2)
    pull = (pull * reg_mask).sum() / reg_mask.sum().clamp(min=1)

    # Push損失：異なる物体同士のembedding（の平均値）を，margin以上離す
    diff = (mean_e.unsqueeze(2) - mean_e.unsqueeze(1)).abs()  # (B, N, N)
    push_raw = F.relu(margin - diff)
    pair_mask = reg_mask.unsqueeze(2) * reg_mask.unsqueeze(1)
    pair_mask = pair_mask * (1 - torch.eye(reg_mask.size(1), device=reg_mask.device))  # 自分自身との組は除外
    push = (push_raw * pair_mask).sum(dim=(1, 2)) / pair_mask.sum(dim=(1, 2)).clamp(min=1)

    return pull, push.mean()


def offset_loss(tl_off, br_off, tl_off_target, br_off_target, tl_ind, br_ind, reg_mask):
    '''GTコーナー位置におけるoffset回帰のSmooth L1損失'''
    tl_pred = gather_feature(tl_off, tl_ind)  # (B, max_objs, 2)
    br_pred = gather_feature(br_off, br_ind)
    mask = reg_mask.unsqueeze(-1)

    tl_loss = (F.smooth_l1_loss(tl_pred, tl_off_target, reduction='none') * mask).sum()
    br_loss = (F.smooth_l1_loss(br_pred, br_off_target, reduction='none') * mask).sum()
    num = reg_mask.sum().clamp(min=1)
    return (tl_loss + br_loss) / num


class CornerNetLoss(nn.Module):
    def __init__(self, pull_weight=0.1, push_weight=0.1, offset_weight=1.0):
        super().__init__()
        self.pull_weight = pull_weight
        self.push_weight = push_weight
        self.offset_weight = offset_weight

    def forward(self, preds, targets):
        tl_heat, br_heat, tl_emb, br_emb, tl_off, br_off = preds
        tl_heat_t, br_heat_t, tl_off_t, br_off_t, tl_ind, br_ind, reg_mask = targets

        heat_loss = focal_loss(tl_heat, tl_heat_t) + focal_loss(br_heat, br_heat_t)
        pull, push = pull_push_loss(tl_emb, br_emb, tl_ind, br_ind, reg_mask)
        off_loss = offset_loss(tl_off, br_off, tl_off_t, br_off_t, tl_ind, br_ind, reg_mask)

        total = heat_loss + self.pull_weight * pull + self.push_weight * push + self.offset_weight * off_loss
        loss_dict = {'heatmap': heat_loss.item(), 'pull': pull.item(), 'push': push.item(), 'offset': off_loss.item()}
        return total, loss_dict

## ネットワークの作成
`CornerNet`をインスタンス化し，`.to(device)`でGPU上に配置します．最適化手法には，SGDではなく**Adam**（学習率1.25e-4）を用います．ヒートマップベースの検出器は，SSDのようなSGD + momentumよりも，Adamの方が学習初期から安定して収束しやすいことが知られており（CornerNetやCenterNetの原論文・多くの実装でもAdamが標準的に使われています），本ノートブックでもこれに合わせます．

In [ ]:
model = CornerNet()
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1.25e-4)
criterion = CornerNetLoss()

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを8，学習エポック数を20とします．バッチ内の各画像について`build_batch_targets`でヒートマップ・offset・embedding抽出位置のターゲットを作成し，ネットワークの出力とあわせて損失を計算します．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)
        batch_targets = build_batch_targets(targets)

        preds = model(images)
        loss, loss_dict = criterion(preds, batch_targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（ピーク検出・コーナーのペアリング・NMS）
学習したモデルから最終的な検出結果を得るには，SSDのようなデコード処理ではなく，次の手順を踏みます．

1. **ピーク検出**：ヒートマップ上で，$3\times3$の近傍の中で自分自身が最大値となっている画素（局所最大値）を「コーナー候補」とする．`heatmap == F.max_pool2d(heatmap, 3, stride=1, padding=1)`という等号判定で実現できる，よく使われるテクニックである．
2. **offset補正**：各コーナー候補の整数座標に，対応するoffset予測を加えてサブピクセル位置を復元し，ストライド倍することで元画像上の座標に変換する．
3. **コーナーのペアリング**：クラスごとに，top-leftの候補とbottom-rightの候補それぞれ上位`max_peaks`個を残し，embeddingの値が近い組み合わせを同じ物体とみなして対応付ける．原論文では複数の付加的なルール（距離・アスペクト比によるフィルタなど）を用いた複雑な手順を取っているが，本ノートブックでは**各top-leftに対してembedding距離が最小のbottom-rightを貪欲に選ぶ**という簡略化を行う（詳細は末尾の「オリジナル論文との違い」を参照）．
4. **NMS**：クラスごとに，得られたボックスに対して`torchvision.ops.nms`で重複した検出結果を除去する．

In [ ]:
def get_peaks(heatmap, k=20, threshold=0.1):
    '''heatmap: (n_class, H, W) から，クラスごとに上位k個の局所最大値の(y, x, score)を取り出す'''
    pooled = F.max_pool2d(heatmap.unsqueeze(0), kernel_size=3, stride=1, padding=1).squeeze(0)
    peak_mask = (heatmap == pooled) & (heatmap > threshold)

    peaks = []
    for c in range(heatmap.size(0)):
        ys, xs = torch.where(peak_mask[c])
        scores = heatmap[c, ys, xs]
        if scores.numel() == 0:
            peaks.append((ys, xs, scores))
            continue
        topk = min(k, scores.numel())
        scores, order = scores.topk(topk)
        peaks.append((ys[order], xs[order], scores))
    return peaks


def predict(model, image_tensor, score_threshold=0.3, embedding_threshold=0.5, nms_threshold=0.5, max_peaks=20):
    model.eval()
    with torch.no_grad():
        preds = model(image_tensor.unsqueeze(0).to(device))
    tl_heat, br_heat, tl_emb, br_emb, tl_off, br_off = [p[0] for p in preds]
    tl_emb, br_emb = tl_emb[0], br_emb[0]  # (H, W)

    tl_peaks = get_peaks(tl_heat, k=max_peaks, threshold=score_threshold)
    br_peaks = get_peaks(br_heat, k=max_peaks, threshold=score_threshold)

    boxes, scores, labels = [], [], []
    for c in range(NUM_HEATMAP_CLASSES):
        tl_ys, tl_xs, tl_scores = tl_peaks[c]
        br_ys, br_xs, br_scores = br_peaks[c]
        if tl_scores.numel() == 0 or br_scores.numel() == 0:
            continue

        # サブピクセルオフセットを加えたのち，元画像座標（ピクセル）にスケールする
        tl_x = (tl_xs.float() + tl_off[0, tl_ys, tl_xs]) * STRIDE
        tl_y = (tl_ys.float() + tl_off[1, tl_ys, tl_xs]) * STRIDE
        br_x = (br_xs.float() + br_off[0, br_ys, br_xs]) * STRIDE
        br_y = (br_ys.float() + br_off[1, br_ys, br_xs]) * STRIDE

        tl_e = tl_emb[tl_ys, tl_xs]
        br_e = br_emb[br_ys, br_xs]

        # 貪欲法によるペアリング：各top-leftに対してembedding距離が最小のbottom-rightを選ぶ
        dist = (tl_e.unsqueeze(1) - br_e.unsqueeze(0)).abs()  # (n_tl, n_br)
        used_br = set()
        for i in range(tl_scores.numel()):
            j = int(torch.argmin(dist[i]).item())
            if dist[i, j].item() > embedding_threshold or j in used_br:
                continue
            x1, y1, x2, y2 = tl_x[i].item(), tl_y[i].item(), br_x[j].item(), br_y[j].item()
            if x2 <= x1 or y2 <= y1:  # 座標が壊れている（矛盾した）組は除外
                continue
            used_br.add(j)
            boxes.append([x1, y1, x2, y2])
            scores.append(((tl_scores[i] + br_scores[j]) / 2).item())
            labels.append(c + 1)  # データセット側の規約（背景=0）に合わせて+1する

    if len(boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    boxes = torch.tensor(boxes, dtype=torch.float32)
    scores = torch.tensor(scores, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.long)

    # クラスごとにNMSを適用して重複した検出を除去する
    keep_boxes, keep_scores, keep_labels = [], [], []
    for c in labels.unique():
        mask = labels == c
        keep = nms(boxes[mask], scores[mask], nms_threshold)
        keep_boxes.append(boxes[mask][keep])
        keep_scores.append(scores[mask][keep])
        keep_labels.append(labels[mask][keep])

    return torch.cat(keep_boxes), torch.cat(keep_scores), torch.cat(keep_labels)

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．mAPの算出は`torchmetrics`の`MeanAveragePrecision`を利用します（詳細は`ssd.ipynb`を参照）．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します。

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します。

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルCornerNetとの違い
本ノートブックで実装したCornerNetは，原論文の構成をいくつかの点で簡略化・変更しています．主な違いは次の通りです．

| | 原論文（CornerNet） | 本ノートブック |
|---|---|---|
| バックボーン | スクラッチ学習のStacked Hourglass Network（砂時計型ネットワークを2つ直列に接続し，中間で教師信号を与えるIntermediate Supervisionも使用） | ImageNet事前学習済みResNet50 + 転置畳み込み3層によるアップサンプリング（Intermediate Supervisionなし） |
| 出力ストライド | 4（Hourglassの出力そのまま） | 4（ResNet50のstride 32出力を，Deconv 3層でstride 4まで戻す） |
| 入力画像サイズ | 511×511（出力128×128） | 256×256（出力64×64） |
| Corner Poolingの実装位置 | Hourglassの出力に対して独自モジュールを適用 | アップサンプリング後の共通特徴マップに対して，`torch.cummax`ベースの自前実装を適用（考え方は同一） |
| コーナーのペアリング | embedding距離・複数の付加的ルール（同一クラス限定，距離の閾値，バックボーンの複数解像度を使った補正など）を組み合わせた手順 | 各top-leftコーナーに対して，embedding距離が最小のbottom-rightコーナーを貪欲に選択する簡略化した手順 |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`を使用 |
| mAP評価 | 独自のCOCO形式評価スクリプト | `torchmetrics`の`MeanAveragePrecision` |

**バックボーンの置き換えについて**：Stacked Hourglassは，物体検出のためだけにスクラッチから数百エポック学習する非常に大きなネットワークであり，本シリーズの他ノートブック（分類・検出）で採用している「事前学習済みバックボーンの再利用」という方針とは相容れません．一方で，CornerNetのようなヒートマップベースの手法は，Hourglassの持つ「高解像度出力」という性質に強く依存しています．そこで本ノートブックでは，ImageNet分類で事前学習されたResNet50から得られる強力な特徴表現を活かしつつ，Deconv層でHourglass同等の高解像度出力を作り出す，というハイブリッドな設計を採用しました．この設計は，本シリーズの検出ノートブック群の中でも特に原論文からの逸脱が大きい変更点です．なお，この「ImageNet事前学習済みバックボーン + Deconvによるアップサンプリング」という構成は，CornerNetの後継であるCenterNetの公式実装においても，Hourglass以外の選択肢（ResNetベースのバリエーション）として実際に採用されているパターンです．

## 課題

1. `pull_push_loss`の`margin`（Push損失のマージン）を変更し，embeddingの学習やコーナーのペアリング精度への影響を確認しましょう．

2. 学習済みモデルに対して，テスト画像1枚のtop-left・bottom-rightヒートマップ（`predict`関数内の`tl_heat`・`br_heat`）を`plt.imshow`でクラスごとに可視化し，コーナー位置にどの程度ピークが立っているか観察しましょう．

3. `CornerHead`のCorner Pooling（`corner_pool_tl`・`corner_pool_br`）を，通常の畳み込み（例えば同じ受容野を持つ畳み込み層）に置き換えたモデルを作り，Corner Poolingの有無で検出精度（`map_50`）がどう変化するか比較しましょう．